# Train on your own labeled rows with a temporal query

Learn how to supply labeled rows, row weights and a train, validation and test split instead of letting Kumo generate the training table.
This Kumo Engine module uses synthetic retail data and a temporal query: for each customer at each anchor time, will they transact in the next 30 days?
It inspects the accepted rows, reads the report for skipped rows, fits a demonstration model and scores customers.

Use a Linux notebook prepared with the [Kumo Engine setup](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/installation/linux-machine-deployment). A GPU is recommended for the FAST training preset; set `SMOKE_TEST = True` below for a quick check on a CPU-only machine.
Expect under a minute on a single L4 GPU after installation.

See [Training tables](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/training-tables#use-custom-training-tables) for the supplied-column contract. For a static version, see the published custom training table notebook.

## Install

Install Kumo Engine in this notebook kernel.

In [1]:
%pip install kumo-engine

## Choose a workspace

Set `KUMO_LOCAL_SSD` to a local SSD mount for faster scratch storage; otherwise the system temporary directory is used.
This cell creates a new directory on each run, with separate source and scratch directories, and does not replace existing data.
The last lines print the installed Kumo Engine version and the workspace.

In [2]:
import importlib.metadata
import os
import tempfile
from pathlib import Path

import json

import numpy as np
import pandas as pd
from kumoapi.model_plan import RunMode
from kumoapi.train import TrainingTableSpec
from kumoapi.typing import Stype
from kumo.engine.trainer.output_config import TrainingTableOutputConfig

import kumo.engine as kumo

workspace_root = Path(os.environ.get("KUMO_LOCAL_SSD", tempfile.gettempdir()))
workspace = Path(tempfile.mkdtemp(prefix="kumo-custom-training-", dir=workspace_root))
source_dir = workspace / "source"
source_dir.mkdir()
kumo.configure(scratch_dir=workspace / "scratch")
print("Kumo Engine:", importlib.metadata.version("kumo-engine"))
print("Workspace:", workspace)

Kumo Engine: 1.0.1
Workspace: /tmp/kumo-custom-training-bo3ui78l


## Set the training budget

Models train with the [FAST preset](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/model-plan-guide).
Set `SMOKE_TEST = True` to cap training at two short epochs for a quick check on a CPU-only machine; its scores carry no meaning.

In [3]:
SMOKE_TEST = False

## Generate synthetic example data

Run the next cell to create synthetic customer and transaction tables: 500 customers and 1,500 transactions over six months.
You can skim the generation code; the Kumo workflow starts below.

In [4]:
rng = np.random.default_rng(0)
customers_frame = pd.DataFrame({"CUSTOMER_ID": range(500), "AGE": rng.integers(20, 80, 500)})
transactions_frame = pd.DataFrame({
    "TX_ID": range(1500),
    "CUSTOMER_ID": rng.integers(0, 500, 1500),
    "TS": pd.Timestamp("2024-01-01") + pd.to_timedelta(rng.integers(0, 180, 1500), unit="D"),
    "AMOUNT": rng.gamma(2.0, 40.0, 1500).round(2),
})
customers_frame.to_parquet(source_dir / "customers.parquet", index=False)
transactions_frame.to_parquet(source_dir / "transactions.parquet", index=False)

## Build the graph

Register each table's primary key, the transaction timestamp and the relationship from `transactions.CUSTOMER_ID` to `customers.CUSTOMER_ID`.
See [Graphs](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/set-up-a-graph/graphs).

In [5]:
connector = kumo.FileConnector(root_dir=str(source_dir))
customers = kumo.Table.from_source_table(
    connector["customers.parquet"], primary_key="CUSTOMER_ID"
).infer_metadata()
transactions = kumo.Table.from_source_table(
    connector["transactions.parquet"], primary_key="TX_ID", time_column="TS"
).infer_metadata().with_column("CUSTOMER_ID", stype=Stype.ID)
graph = kumo.Graph(
    tables={"customers": customers, "transactions": transactions},
    edges=[kumo.Edge(src_table="transactions", fkey="CUSTOMER_ID", dst_table="customers")],
)
graph.validate()
snapshot = graph.snapshot()

## Define the query

The query is temporal because the label looks forward 30 days from an anchor time, so each supplied row is a customer at an anchor.
The task type should be `binary_classification`. See [Predictive Queries](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/predictive-queries).

In [6]:
query = kumo.PredictiveQuery(
    graph, "PREDICT COUNT(transactions.*, 0, 30, days) > 0 FOR EACH customers.CUSTOMER_ID"
)
print("Task type:", query.get_task_type())
print(query.validate())

Task type: binary_classification
ValidationResponse(warnings=[], errors=[], info_items=[])


## Supply labels, weights and splits

| Column | Meaning |
| --- | --- |
| `ENTITY1` | The customer's key in the graph |
| `TIMESTAMP` | Anchor time, timezone-naive: features use data before it, the label uses the 30 days after it |
| `TARGET1` | The label for that customer at that anchor |
| `SPLIT` | Optional, any name: `0` train, `1` validation, `2` test |
| `WEIGHT` | Optional, any name: each row's contribution when weighted loss is selected |

The names `ENTITY1`, `TIMESTAMP` and `TARGET1` are part of the custom-table contract, not the graph's column names.
Here each label is computed from the transactions, the split follows time so the latest anchors are held out, and positive rows get double weight.

In [7]:
def labels_at(anchor):
    window = transactions_frame[
        (transactions_frame.TS >= anchor) & (transactions_frame.TS < anchor + pd.Timedelta(days=30))
    ]
    customers_sampled = rng.choice(500, 150, replace=False)
    return pd.DataFrame({
        "ENTITY1": customers_sampled,
        "TIMESTAMP": anchor,
        "TARGET1": np.isin(customers_sampled, window.CUSTOMER_ID).astype(int),
    })


anchors = pd.date_range("2024-02-01", "2024-04-01", freq="14D")
labeled_rows = pd.concat([labels_at(anchor) for anchor in anchors], ignore_index=True)
labeled_rows["SPLIT"] = np.select(
    [labeled_rows.TIMESTAMP < "2024-03-01", labeled_rows.TIMESTAMP < "2024-03-15"], [0, 1], default=2
)
labeled_rows["WEIGHT"] = np.where(labeled_rows.TARGET1 == 1, 2.0, 1.0)
labeled_rows.to_parquet(source_dir / "training.parquet", index=False)
print(labeled_rows.groupby("SPLIT").TARGET1.agg(rows="size", positive_rate="mean"))

       rows  positive_rate
SPLIT                     
0       450       0.377778
1       150       0.380000
2       150       0.386667


## Inspect the accepted rows

Materialize the rows against the snapshot, export what Kumo accepted and count its split assignments. Expect all 750 rows with the same split counts as above.
Keep the `TrainingTableSpec`; it is passed to model-plan suggestion later so the plan accounts for the weight column.

In [8]:
spec = TrainingTableSpec(weight_col="WEIGHT", split_col="SPLIT")
training_table = kumo.TrainingTable.from_source_table(
    connector["training.parquet"], spec, pq=query, snapshot=snapshot
)
training_table.export(TrainingTableOutputConfig(
    connector=kumo.FileConnector(root_dir=str(workspace / "outputs")),
    table_name="accepted_training_rows",
))
accepted_rows = pd.read_parquet(workspace / "outputs" / "accepted_training_rows")
print("Accepted rows:", len(accepted_rows))
print(accepted_rows.groupby("SPLIT").size().rename("rows"))

Accepted rows: 750
SPLIT
0    450
1    150
2    150
Name: rows, dtype: int64


## Recognize skipped rows

Rows whose split is not `0`, `1` or `2` are skipped, not rejected. Corrupt the first 21 split values and read the row-failure report in the table's manifest.
A successful materialization does not mean every source row was accepted, and a few skipped rows usually point to an upstream data bug.

In [9]:
def materialize(frame, name):
    frame.to_parquet(source_dir / name, index=False)
    return kumo.TrainingTable.from_source_table(connector[name], spec, pq=query, snapshot=snapshot)


with_invalid_split = labeled_rows.assign(SPLIT=np.where(labeled_rows.index < 21, 9, labeled_rows.SPLIT))
skipped = materialize(with_invalid_split, "invalid_split.parquet")
row_failures = json.loads(skipped.manifest_path.read_text())["row_failures"]
print({key: row_failures[key] for key in ("rows_read", "skipped_rows")})
print([(c["condition"], c["column"], c["count"]) for c in row_failures["by_condition"]])

{'rows_read': 750, 'skipped_rows': 21}
[('invalid_split', 'SPLIT', 21)]


If no row has a valid split, here because the split holds strings, nothing is left to train on and the call raises.
A missing required column also raises, naming the column.

In [10]:
as_strings = labeled_rows.assign(SPLIT=labeled_rows.SPLIT.map({0: "train", 1: "val", 2: "test"}))
for name, frame in {
    "string_split.parquet": as_strings,
    "without_timestamp.parquet": labeled_rows.drop(columns="TIMESTAMP"),
}.items():
    try:
        materialize(frame, name)
    except Exception as error:
        print(f"{type(error).__name__}: {error}\n")

DiskgraphDataError: custom-TT source kept no rows (750 read, 750 skipped), so the run would train on nothing

ValueError: Required column not found. Source table is missing TIMESTAMP, which is the name DiskGraph reads that column under and the only name it reads it under. It carries ['ENTITY1', 'SPLIT', 'TARGET1', 'WEIGHT'].



## Fit a small model

Fit the original table with its explicit snapshot. The FAST plan enables weighted loss and uses one experiment without refitting; refits and explanations are disabled to keep this module on supplied training rows.
See [Training](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/training). Metrics from synthetic data demonstrate that the table reaches training; use a proper evaluation budget for your own problem.

In [11]:
def fast_plan(query, snapshot, **suggest_options):
    model_plan = query.suggest_model_plan(snapshot, run_mode=RunMode.FAST, **suggest_options)
    model_plan.training_job.num_experiments = 1
    model_plan.training_job.refit_trainval = False
    model_plan.training_job.refit_full = False
    model_plan.training_job.disable_explain = True
    if SMOKE_TEST:
        model_plan.optimization.max_epochs = 2
        model_plan.optimization.max_steps_per_epoch = 20
    return model_plan


model_plan = fast_plan(query, snapshot, train_table_spec=spec)
model_plan.optimization.weight_mode = ["weighted_loss"]
trainer = kumo.Trainer(model_plan=model_plan)
fit_result = trainer.fit(training_table, snapshot=snapshot)
print("Synthetic-data metrics: demonstration only")
print(pd.Series(fit_result.metrics, name="demonstration_metric"))

Synthetic-data metrics: demonstration only
val_loss       0.849413
val_ap         0.457685
val_auprc      0.445924
val_auroc      0.561121
test_loss      0.812830
test_ap        0.385132
test_auprc     0.373907
test_auroc     0.494471
train_loss     0.524225
train_ap       0.874019
train_auprc    0.874072
train_auroc    0.908535
Name: demonstration_metric, dtype: float64


## Score customers

Score the default population for the latest anchor. Passing the trained model's `training_table_generation_plan` keeps the prediction table consistent with the fitted model.
Expect one row per customer; `True_PROB` is the probability of a transaction in the next 30 days. See [Making predictions](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/making-predictions).

In [12]:
prediction_table = query.generate_prediction_table(
    snapshot, plan=trainer.training_table_generation_plan
)
scores = trainer.predict(
    prediction_table, snapshot=snapshot, binary_classification_threshold=0.5
).frame
print("Scored customers:", len(scores))
scores.head(3)

Scored customers: 500


,ENTITY,TARGET_PRED,False_PROB,True_PROB
0,31,False,0.599777,0.400223
1,34,False,0.562581,0.437419
2,37,False,0.699464,0.300536


## Next step

Build the labeled rows from your own history, choose split cutoffs that keep future information out of training, and pass the same `TrainingTableSpec` to `TrainingTable.from_source_table`, model-plan suggestion and `Trainer.fit`.
To choose which rows to score, see [custom prediction table](local_kumo_engine_custom_prediction_table.ipynb); for query design, see [static query predictions](local_kumo_engine_static_query_predictions.ipynb).

## Cleanup

Remove this notebook's workspace, which holds only the files it created. Skip this cell to keep the generated tables and predictions.

In [13]:
import shutil

shutil.rmtree(workspace)